# Unit 1: Working with Audio Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

**Course**: [HuggingFace Audio Transformers Course - Unit 1](https://huggingface.co/learn/audio-course/chapter1)

**Topics covered:**
- Audio fundamentals: waveforms, sampling rate, amplitude, bit depth
- Spectrograms and mel spectrograms
- Loading and preprocessing audio datasets with `datasets`
- Streaming large audio datasets
- Resampling and preparing audio for models

## Setup

In [ ]:
!pip install -q transformers datasets librosa soundfile matplotlib numpy IPython

## 1. Audio Fundamentals

Sound is a continuous wave. To store it digitally, we **sample** the wave at regular intervals.

Key concepts:
- **Sampling rate**: Number of samples per second (Hz). CD quality = 44,100 Hz. Speech models often use 16,000 Hz.
- **Amplitude**: Height of the wave at each sample point.
- **Bit depth**: Precision of each sample (e.g., 16-bit = 65,536 possible values).

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import IPython.display as ipd

# Load a sample audio file from librosa
audio_array, sr = librosa.load(librosa.ex('trumpet'), sr=None)

print(f"Sampling rate: {sr} Hz")
print(f"Audio shape: {audio_array.shape}")
print(f"Duration: {len(audio_array) / sr:.2f} seconds")
print(f"Dtype: {audio_array.dtype}")

# Listen to it
ipd.Audio(audio_array, rate=sr)

### Visualizing a Waveform

In [ ]:
plt.figure(figsize=(14, 4))
librosa.display.waveshow(audio_array, sr=sr)
plt.title("Waveform")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.tight_layout()
plt.show()

### Effect of Sampling Rate

Resampling to a lower rate reduces quality but also file size / sequence length.

In [ ]:
# Resample to 16kHz (common for speech models)
audio_16k = librosa.resample(audio_array, orig_sr=sr, target_sr=16000)

print(f"Original: {len(audio_array)} samples at {sr} Hz")
print(f"Resampled: {len(audio_16k)} samples at 16000 Hz")

ipd.Audio(audio_16k, rate=16000)

## 2. Spectrograms

A **spectrogram** shows how frequencies change over time. Created via Short-Time Fourier Transform (STFT).

A **mel spectrogram** uses the mel scale, which better matches human perception of pitch.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Standard spectrogram
S = librosa.stft(audio_array)
S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)
librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='hz', ax=axes[0])
axes[0].set_title('Spectrogram')

# Mel spectrogram
S_mel = librosa.feature.melspectrogram(y=audio_array, sr=sr, n_mels=128)
S_mel_db = librosa.power_to_db(S_mel, ref=np.max)
librosa.display.specshow(S_mel_db, sr=sr, x_axis='time', y_axis='mel', ax=axes[1])
axes[1].set_title('Mel Spectrogram')

plt.tight_layout()
plt.show()

## 3. Loading Audio Datasets with HuggingFace

The `datasets` library makes it easy to load audio datasets from the Hub.

In [ ]:
from datasets import load_dataset

# Load a small speech dataset
dataset = load_dataset("PolyAI/minds14", name="en-US", split="train", trust_remote_code=True)
print(dataset)
print(f"\nFeatures: {dataset.features}")
print(f"\nFirst example keys: {dataset[0].keys()}")

In [ ]:
# Examine an audio sample
example = dataset[0]
audio = example["audio"]

print(f"Sampling rate: {audio['sampling_rate']}")
print(f"Array shape: {np.array(audio['array']).shape}")
print(f"Transcript: {example.get('transcription', 'N/A')}")

ipd.Audio(audio["array"], rate=audio["sampling_rate"])

### Resampling a Dataset

Many models expect a specific sampling rate. Use `cast_column` to resample on the fly.

In [ ]:
from datasets import Audio

# Resample entire dataset to 16kHz
dataset_16k = dataset.cast_column("audio", Audio(sampling_rate=16000))

example_16k = dataset_16k[0]
print(f"New sampling rate: {example_16k['audio']['sampling_rate']}")
print(f"New array shape: {np.array(example_16k['audio']['array']).shape}")

### Streaming Large Datasets

For large datasets, use `streaming=True` to avoid downloading everything upfront.

In [ ]:
# Stream a dataset
streamed_dataset = load_dataset("PolyAI/minds14", name="en-US", split="train", streaming=True)

# Iterate through first few examples
for i, example in enumerate(streamed_dataset):
    print(f"Example {i}: sampling_rate={example['audio']['sampling_rate']}, "
          f"duration={len(example['audio']['array'])/example['audio']['sampling_rate']:.2f}s")
    if i >= 4:
        break

## 4. Preprocessing Audio for Models

HuggingFace feature extractors handle the preprocessing (resampling, normalization, spectrogram creation).

In [ ]:
from transformers import WhisperFeatureExtractor

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-tiny")

# Process an audio sample
example = dataset_16k[0]
inputs = feature_extractor(
    example["audio"]["array"],
    sampling_rate=example["audio"]["sampling_rate"],
    return_tensors="pt"
)

print(f"Input features shape: {inputs.input_features.shape}")
print(f"This is a log-mel spectrogram: (batch, mel_bins, time_steps)")

In [ ]:
# Visualize the Whisper input features
plt.figure(figsize=(14, 4))
plt.imshow(inputs.input_features[0].numpy(), aspect='auto', origin='lower')
plt.colorbar(label='Log-Mel Energy')
plt.title('Whisper Input Features (Log-Mel Spectrogram)')
plt.xlabel('Time Steps')
plt.ylabel('Mel Bins')
plt.tight_layout()
plt.show()

## Key Takeaways

- Audio is sampled at a fixed rate; common rates are 16kHz (speech) and 44.1kHz (music)
- Spectrograms convert time-domain signals to time-frequency representations
- Mel spectrograms align better with human hearing perception
- HuggingFace `datasets` handles audio loading, resampling, and streaming
- Feature extractors from `transformers` handle model-specific preprocessing

**Next**: [Unit 2 - Audio Applications with Pipelines](../unit_2/02_audio_applications_pipelines.ipynb)